# DSC 540 – Milestone 3: Cleaning and Formatting Website Data

**Student:** Anil Bammidi
**Milestone:** Week 7 & 8 – Milestone 3
**Date:** 02/15/2026 (revised)

## What this notebook delivers
- Data pulled **directly from the source URL** (no CSV export), with certificate verification enabled.
- Labeled transformation steps (**Step 1 through Step 7**), each with a short description.
- A derived `days_to_pay` measure and **flags** for outliers and inconsistent dates. Flagged rows are kept, not deleted.
- A human-readable cleaned dataset, saved to `cleaned_prompt_payment.csv`.
- An ethics paragraph that describes only the changes the notebook actually makes.

**Data source:** Government of Canada Prompt Payment / Payment Delay
**Source URL:** `https://donnees-data.tpsgc-pwgsc.gc.ca/br1/delaipaiement-promptpayment/delaipaiement-promptpayment.json`

**Revisions from the first submission:**
- Removed the `verify=False` SSL fallback. Certificate verification is always on.
- Title Case is applied only to vendor names, in a new column. Identifiers and project numbers are no longer changed.
- Removed the unused API-token step (the endpoint is public).
- Added the `days_to_pay` measure so the outlier check runs on real numeric data.
- Removed the Airbnb HTML scraping section. Airbnb's terms restrict automated access, so that section should not be part of this submission.

In [ ]:
import json
import requests
import certifi
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)

## Step 1 – Pull raw data from the source URL
Requests the JSON from the government endpoint and loads it into a DataFrame. Certificate verification uses the certifi CA bundle. If the request fails, the notebook stops with an error rather than falling back to an unverified connection.

In [ ]:
SERVICE_URL = "https://donnees-data.tpsgc-pwgsc.gc.ca/br1/delaipaiement-promptpayment/delaipaiement-promptpayment.json"
HEADERS = {"User-Agent": "DSC540-Milestone3/1.0"}

resp = requests.get(SERVICE_URL, headers=HEADERS, timeout=60, verify=certifi.where())
resp.raise_for_status()
raw = resp.json()

# The JSON may be a list of records or a dict that wraps the list.
if isinstance(raw, list):
    records = raw
else:
    records = next((raw[k] for k in ["data", "records", "result", "items"]
                    if isinstance(raw.get(k), list)), [])

df_raw = pd.DataFrame(records)
print("Raw shape:", df_raw.shape)
df_raw.head()

## Step 2 – Clean headers
Strips whitespace, replaces underscores with spaces, and collapses repeated spaces. The original data is kept in `df_raw`.

In [ ]:
df = df_raw.copy()
df.columns = (df.columns
              .astype(str)
              .str.strip()
              .str.replace("_", " ", regex=False)
              .str.replace(r"\s+", " ", regex=True))
print(df.columns.tolist())

## Step 3 – Handle missing values
Placeholder strings (`""`, `N/A`, `NA`, `null`, `None`) become true missing values. Missing text is filled with `"Unknown"` so it stays visible. Missing dates are left empty, because a made-up date would distort the payment-delay measure. The counts before and after are printed.

In [ ]:
PLACEHOLDERS = ["", "N/A", "NA", "null", "None"]
df = df.replace(PLACEHOLDERS, np.nan)

missing_before = int(df.isna().sum().sum())

text_cols = [c for c in df.select_dtypes(include=["object"]).columns if "date" not in c.lower()]
for c in text_cols:
    df[c] = df[c].fillna("Unknown")

print("Missing values before:", missing_before)
print("Missing values after (text filled; dates left empty):", int(df.isna().sum().sum()))

## Step 4 – Convert data types (dates)
Converts the two date columns to datetime, using their exact column names. Values that cannot be parsed become empty and are counted, instead of being silently changed.

In [ ]:
DATE_COLS = [
    "Payment-date Date-de-paiement",
    "Proper-Invoice-Received-Date date-de-réception-de-la-facture-en-règle",
]

for c in DATE_COLS:
    before_valid = df[c].ne("Unknown").sum()
    parsed = pd.to_datetime(df[c].where(df[c].ne("Unknown")), errors="coerce")
    failed = int(parsed.isna().sum() - df[c].eq("Unknown").sum())
    df[c] = parsed
    print(f"{c}: {failed} values could not be parsed")

print(df.dtypes)

## Step 5 – Normalize text (whitespace and vendor names)
Trims and collapses whitespace in all text columns. Vendor names also get a Title Case version in a new column, `Vendor-name (cleaned)`. The original vendor name is kept. Identifiers and project numbers are not changed.

In [ ]:
for c in df.select_dtypes(include=["object"]).columns:
    df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

VENDOR = "Vendor-name Nom-du-fournisseur"
df["Vendor-name (cleaned)"] = df[VENDOR].str.title()

df[[VENDOR, "Vendor-name (cleaned)"]].head()

## Step 6 – Remove exact duplicates
Removes rows that are identical in every column. Rows that share an ID but differ in any field (for example, a different project number or invoice date) are kept, because they may be separate payments.

In [ ]:
before = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Removed {before - len(df)} exact duplicate rows. Remaining rows: {len(df)}")

## Step 7 – Derive payment delay and flag outliers and inconsistencies
Computes `days_to_pay` as payment date minus the date the proper invoice was received. Then it flags:
- **negative_days:** payment dated before the invoice was received (inconsistent record)
- **outlier_days:** `days_to_pay` outside the IQR fences (1.5 × IQR beyond the quartiles)

Flags are added as columns. No rows are removed.

In [ ]:
PAY = "Payment-date Date-de-paiement"
INV = "Proper-Invoice-Received-Date date-de-réception-de-la-facture-en-règle"

df["days_to_pay"] = (df[PAY] - df[INV]).dt.days

q1, q3 = df["days_to_pay"].quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr

df["negative_days"] = df["days_to_pay"] < 0
df["outlier_days"] = (df["days_to_pay"] < lower) | (df["days_to_pay"] > upper)

print("days_to_pay summary:")
print(df["days_to_pay"].describe())
print(f"IQR fences: {lower:.0f} to {upper:.0f} days")
print("Rows with negative_days:", int(df["negative_days"].sum()))
print("Rows with outlier_days:", int(df["outlier_days"].sum()))

## Final cleaned dataset
Prints the shape and a preview, then saves the dataset to `cleaned_prompt_payment.csv`.

In [ ]:
print("Final cleaned dataset shape:", df.shape)
df.to_csv("cleaned_prompt_payment.csv", index=False)
df.head(25)

## Ethical Implications of Data Wrangling
The data comes from a public Government of Canada endpoint. It records vendor names, project numbers, and procurement IDs, but no individual personal information, so privacy risk is low. The main ethical concern is accuracy. The notebook changes headers and whitespace, replaces placeholder values with "Unknown," parses dates, and removes only exact duplicate rows. It adds a Title Case vendor name as a separate column and leaves the original values in place. It does not impute missing dates, so the payment-delay measure is not made up for incomplete records. Outliers and negative delays are flagged rather than removed, since a long delay can be real. The assumption that a payment dated before its invoice is inconsistent may not hold for every record, so those flags need review before any conclusion is drawn. Official-source credibility is not validated beyond the structure of the records, and the dataset is a snapshot of the source at the time it was retrieved. To mitigate these risks, the raw data is kept unchanged, every step is labeled and reproducible, and the cleaned file records the flags alongside the original values.